In [163]:
import os
import sys
import multiprocessing


import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pyProBound_operator as pbo

import logomaker

In [295]:
def hamming_distance(s1, s2):
    """
    Calculates the Hamming distance between two strings s1 and s2.
    """
    return sum(c1 != c2 for c1, c2 in zip(s1, s2))

def find_similar_strings(input_str, strings):
    """
    Finds strings in the list `strings` that have a Hamming distance of up to 2
    from the input string `input_str`.
    """
    similar_strings = []
    for s in strings:
        if hamming_distance(input_str, s) < 2:
            similar_strings.append(True)
        else:
            similar_strings.append(False)
    return similar_strings 

In [285]:
path = '/home/gralak/updepla/users/gralak/NAS2/SmileSeq_paper/SmileSeq_experiments/inputs/20220915_input/00_read_in_data/output/'
#path = '/home/gralak/updepla/users/gralak/NAS2/SmileSeq_paper/SmileSeq_experiments/SmSAG23/00_read_in_data/output/'


In [286]:
files = os.listdir(path)

In [296]:
mBC = 'AGTA'
nBC = 'GAAT'

In [288]:
wrongly_assigned_files = dict()

for file in files:
    df = pd.read_csv(path + file)
    id = file.split('_')[0]
    # input first, change all mBCs with hamming dsitance 2 or less into the corresponding mBCs.
    mBCs = df['methl']

    #identify all mBC which are hamming distance 2 from AGTA to AGTA
    similar_strings_m = find_similar_strings(mBC, mBCs)
    true_hits_m = list(mBCs == mBC)

        #input_df.loc[similar_strings, 'methl'] = mBC

    #identify all mBC which are hamming distance 2 from GAAT to GAAT
    similar_strings_nm = find_similar_strings(nBC, mBCs)
    true_hits_nm = list(mBCs == nBC)

    #extract the rows from df
    hamming_m = df[similar_strings_m]
    #true_m = df[true_hits_m]
    hamming_nm = df[similar_strings_nm]
    #true_nm = df[true_hits_nm]

    #identify the overlap
    ll_m = []
    ll_nm = []
    for BC1, _ in hamming_m.groupby(by='methl'):
        ll_m.append(BC1)

    for BC2, _ in hamming_nm.groupby(by='methl'):
        ll_nm.append(BC2)
    
    common_elements = list(set(ll_m) & set(ll_nm))
    
    #assess the damage
    wrongly_assigned = 0
    for seq in common_elements:
        wrongly_assigned += len(df[df['methl'] == seq])
    
    # in percent
    in_perc = wrongly_assigned / len(df)

    wrongly_assigned_files[id] = [wrongly_assigned, in_perc]
    
print(wrongly_assigned_files)

{'BC5': [0, 0.0], 'BC11': [0, 0.0], 'BC7': [0, 0.0], 'BC2': [0, 0.0], 'BC12': [0, 0.0], 'BC10': [0, 0.0], 'BC6': [0, 0.0], 'BC3': [0, 0.0], 'BC9': [0, 0.0], 'BC1': [0, 0.0], 'BC8': [0, 0.0], 'BC4': [0, 0.0]}


In [304]:
def _hamming_distance(s1, s2):
    """
    Calculates the Hamming distance between two strings s1 and s2.
    """
    return sum(c1 != c2 for c1, c2 in zip(s1, s2))

def _find_similar_strings(input_str, strings):
    """
    Finds strings in the list `strings` that have a Hamming distance of 1
    from the input string `input_str`.
    """
    similar_strings = []
    for s in strings:
        if _hamming_distance(input_str, s) < 2:
            similar_strings.append(True)
        else:
            similar_strings.append(False)
    return similar_strings


def barcode_correction(df, methylation_BC):
    """
    Corrects barcodes in the 'methl' column of the DataFrame by replacing 
    any string within Hamming distance <= 1 of the reference barcodes.

    Parameters:
    - df: pandas DataFrame with a 'methl' column
    - methylated_BC: reference string for methylated barcode (e.g. 'AGTA')
    - unmethylated_BC: reference string for unmethylated barcode (e.g. 'GAAT')

    Returns:
    - df: corrected DataFrame (modified in place)
    """
    mBCs = df['methl']

    similar_to_meth = _find_similar_strings(methylation_BC, mBCs)
    df.loc[similar_to_meth, 'methl'] = methylation_BC

    return df

In [299]:
df

,methl,BC,lfl,random24,rfl
0,GAAT,CACACAA,TA,CTAACGTTATACGATATCTGAACC,TAGAG
1,GAAT,CACACAA,TA,TAATGTTTACTGGTCAACGGTGGA,TAGAG
2,AGTA,CACACAA,TA,CCTTGATAATACATCAGAGGGCAC,TAGAG
3,AGTA,CACACAA,TA,GGCGCTCCGAGTTACCATAAGTTA,TAGAG
4,AGTA,CACACAA,TA,GGACTGTTCGCAATGTTGGGGTCT,TAGAG
...,...,...,...,...,...
26818,AGTA,CACACAA,TA,TTCAAACTGTCTTAGTGCCGTGCA,TAGAG
26819,AGTA,CACACAA,TA,ATTCGACAGTTAGCTTGGGGGACC,TAGAG
26820,AGTA,CACACAA,TA,ACCGGGAATGTACTGAGGCTGCTG,TAGAG
26821,GAAT,CACACAA,TA,TCTCGCAAATAGGGATCGCTTATC,TAGAG


In [300]:
mBCs = df['methl']

#identify all mBC which are hamming distance 2 from AGTA to AGTA
similar_strings_m = find_similar_strings(mBC, mBCs)

In [301]:
len(df[similar_strings_m])

16567

In [305]:
test = barcode_correction(df, mBC)

In [307]:
len(test[test['methl'] == mBC])

16567

In [143]:
# input first, change all mBCs with hamming dsitance 2 or less into the corresponding mBCs.
mBCs = df['methl']

    #change all mBC which are hamming distance 2 from AGTA to AGTA
similar_strings_m = find_similar_strings(mBC, mBCs)

    #input_df.loc[similar_strings, 'methl'] = mBC

    #change all mBC which are hamming distance 2 from GAAT to GAAT
similar_strings_nm = find_similar_strings(nBC, mBCs)

    #input_df.loc[similar_strings, 'methl'] = unmethylated_BC

In [144]:
true_hits_m = list(mBCs == mBC)

In [145]:
hamming_m = df[similar_strings_m]
true_m = df[true_hits_m]

In [146]:
print(len(hamming_m), len(true_m))

603442 601647


In [147]:
true_hits_nm = list(mBCs == nBC)
hamming_nm = df[similar_strings_nm]
true_nm = df[true_hits_nm]

In [148]:
ll_m = []
ll_nm = []
for BC, _ in hamming_m.groupby(by='methl'):
    ll_m.append(BC)

#print('######### methl assigned BCs:')
#print(ll_m)

for BC, _ in hamming_nm.groupby(by='methl'):
    ll_nm.append(BC)

#print('######### nonmethl assigned BCs:')
#print(ll_nm)

In [149]:
common_elements = list(set(ll_m) & set(ll_nm))

print(common_elements)

['AAAA', 'GGAA', 'AGAT', 'GATA', 'AATT']


In [150]:
wrongly_assigned = 0
for seq in common_elements:
    wrongly_assigned += len(df[df['methl'] == seq])
print(wrongly_assigned)

65


# Reverse Complement of some TFs

In [4]:
import os
import pandas as pd

In [11]:
file

'PRDM10_DBD_bm15_methylated_bindingmode_1.csv'

In [ ]:
folder = '/home/gralak/updepla/users/gralak/SmileSeq_paper/meSMiLEseq_separated_analysis/psam_used_in_pub/'
my_files = os.listdir(folder)

for file in my_files:
    if file.endswith('bindingmode_1.csv'):
        psam = pd.read_csv(folder + file, index_col=0)

        #rev complement
        rev_colnames = ['T','G','C','A']
        psam_rev = psam.iloc[::-1]
        psam_rev.columns = rev_colnames
        psam_rev = psam_rev.reset_index(drop=True)
        psam_rev = psam_rev[['A','C','G','T']]

        name = file.replace('.csv', '_revcomp.csv')
        psam_rev.to_csv(folder + name)

In [21]:
file.split('_')

['FAM200B', 'DBD', 'bm15', 'unmethylated', 'bindingmode', '1.csv']

In [27]:
import matplotlib.pyplot as plt
import logomaker

In [31]:
folder = '/home/gralak/updepla/users/gralak/SmileSeq_paper/meSMiLEseq_separated_analysis/psam_used_in_pub/'
out = os.path.join(folder, 'psam_logo/')
my_files = os.listdir(folder)

for file in my_files:
    if file.endswith('bindingmode_1revcomp.csv'):
        name = file.split('_')
        TF = name[0] + '_' + name[1]
        mstat = name[3]
        binding_mode = name[2]
        psam = pd.read_csv(folder + file, index_col=0)

        fig, ax = plt.subplots(1,1,figsize=[10,6])
        logo = logomaker.Logo(psam,
                            shade_below=0.5,
                            ax=ax,
                            fade_below=0.5,
                            color_scheme={'A':'#66a61e', 'C':'#7570b3','G':'#ffc809','T':'#d95f02','m':'#a6cee3'}
                            )
                        # style using Logo methods
        logo.style_spines(visible=False)
        logo.style_spines(spines=['left', 'bottom'], visible=True)
        logo.style_xticks(rotation=90, fmt='%d', anchor=0)

                        # style using Axes methods
        logo.ax.set_ylabel("$-\Delta \Delta G$ (kcal/mol)", labelpad=-1)
        logo.ax.xaxis.set_ticks_position('none')
        logo.ax.xaxis.set_tick_params(pad=-1)
                        #logo.ax.set_ylim([-6, 4])

        fig.suptitle(f"{TF} {mstat} bindingmode 1")

        fig.savefig(f'{out}{TF}_{binding_mode}_{mstat}_bindingmode_1_logo_revcomp.pdf', format='pdf')
        #fig.savefig(f'{out}{TF}_{binding_mode}_{mstat}_bindingmode_1_logo_revcomp.png', format='png')
        plt.close()